# Module 3: DataFrames Introduction

DataFrames are Spark's **primary API** for structured data. They're like RDDs with superpowers:

- **Schema**: Every column has a name and type. Spark knows the structure of your data.
- **Catalyst optimizer**: Spark automatically rewrites your queries for maximum performance — the same way a database optimizer works.
- **Familiar API**: If you know SQL or pandas, you already know 80% of the DataFrame API.

**Bottom line**: In production Spark, you use DataFrames for 95% of everything. RDDs are for rare edge cases.

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Module 03 - DataFrames Intro") \
    .master("local[*]") \
    .getOrCreate()

---
## Concept 1: Creating DataFrames from Python Data

You can create DataFrames from lists of tuples (with column names) or lists of dictionaries. This is useful for testing and prototyping.

In [2]:
# From tuples with column names
data = [
    ("Alice", 30, "San Francisco"),
    ("Bob", 25, "New York"),
    ("Carol", 35, "Chicago"),
]
df = spark.createDataFrame(data, ["name", "age", "city"])
df.show()

+-----+---+-------------+
| name|age|         city|
+-----+---+-------------+
|Alice| 30|San Francisco|
|  Bob| 25|     New York|
|Carol| 35|      Chicago|
+-----+---+-------------+



In [3]:
# From a list of dictionaries — Spark infers column names from keys
data_dicts = [
    {"name": "Dave", "age": 28, "city": "Boston"},
    {"name": "Eve", "age": 32, "city": "Seattle"},
]
df2 = spark.createDataFrame(data_dicts)
df2.show()

+---+-------+----+
|age|   city|name|
+---+-------+----+
| 28| Boston|Dave|
| 32|Seattle| Eve|
+---+-------+----+



#### Try It: Create a Books DataFrame

Create a DataFrame with 5 rows and columns: `title` (string), `author` (string), `year` (int), `pages` (int).

Use `.show()` and `.printSchema()` to inspect it.

In [4]:
# Your code here


In [5]:
# --- Solution ---
books = [
    ("The Great Gatsby", "Fitzgerald", 1925, 180),
    ("To Kill a Mockingbird", "Harper Lee", 1960, 281),
    ("1984", "George Orwell", 1949, 328),
    ("Pride and Prejudice", "Jane Austen", 1813, 432),
    ("The Catcher in the Rye", "J.D. Salinger", 1951, 234),
]
books_df = spark.createDataFrame(books, ["title", "author", "year", "pages"])
books_df.show(truncate=False)
books_df.printSchema()

+----------------------+-------------+----+-----+
|title                 |author       |year|pages|
+----------------------+-------------+----+-----+
|The Great Gatsby      |Fitzgerald   |1925|180  |
|To Kill a Mockingbird |Harper Lee   |1960|281  |
|1984                  |George Orwell|1949|328  |
|Pride and Prejudice   |Jane Austen  |1813|432  |
|The Catcher in the Rye|J.D. Salinger|1951|234  |
+----------------------+-------------+----+-----+

root
 |-- title: string (nullable = true)
 |-- author: string (nullable = true)
 |-- year: long (nullable = true)
 |-- pages: long (nullable = true)



---
## Concept 2: Explicit Schema with StructType

When Spark infers types, it guesses — and sometimes guesses wrong (e.g., reading "123" as a string instead of int). In production, you always define an **explicit schema**.

- `StructType` — the schema (list of columns)
- `StructField` — one column (name, type, nullable)
- Types: `StringType`, `IntegerType`, `FloatType`, `DoubleType`, `BooleanType`, `DateType`, etc.

**Why explicit schemas matter in Big Data**: When you `inferSchema=True` on a 10TB file, Spark reads the ENTIRE file just to guess types before processing it. An explicit schema skips this — making reads 2x faster.

In [6]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType

schema = StructType([
    StructField("name", StringType(), nullable=True),
    StructField("age", IntegerType(), nullable=True),
    StructField("city", StringType(), nullable=True),
])

df3 = spark.createDataFrame(data, schema)
df3.printSchema()  # Notice age is now IntegerType, not LongType

root
 |-- name: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- city: string (nullable = true)



#### Try It: Define a Schema for departments.csv

1. Define a `StructType` schema with columns: `department_id` (IntegerType), `department_name` (StringType), `budget` (IntegerType), `location` (StringType)
2. Read `../data/departments.csv` using this schema (pass `schema=your_schema`)
3. Print the schema and show all rows

In [7]:
# Your code here


In [8]:
# --- Solution ---
dept_schema = StructType([
    StructField("department_id", IntegerType(), False),
    StructField("department_name", StringType(), True),
    StructField("budget", IntegerType(), True),
    StructField("location", StringType(), True),
])

departments = spark.read.csv("../data/departments.csv", header=True, schema=dept_schema)
departments.printSchema()
departments.show()

root
 |-- department_id: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- budget: integer (nullable = true)
 |-- location: string (nullable = true)

+-------------+---------------+-------+-------------+
|department_id|department_name| budget|     location|
+-------------+---------------+-------+-------------+
|          101|    Engineering|2500000|San Francisco|
|          102|      Marketing|1200000|     New York|
|          103|          Sales|1800000|      Chicago|
|          104|Human Resources| 800000|     New York|
|          105|        Finance|1500000|San Francisco|
|          106|     Operations| 950000|      Chicago|
+-------------+---------------+-------+-------------+



---
## Concept 3: Reading CSV Files

The most common way to create DataFrames. Key options:
- `header=True` — first row is column names
- `inferSchema=True` — guess types (convenient but slow on large files)
- `schema=your_schema` — use explicit schema (fast, reliable)

In [9]:
# Read with schema inference (convenient for exploration)
employees = spark.read.csv("../data/employees.csv", header=True, inferSchema=True)
departments = spark.read.csv("../data/departments.csv", header=True, inferSchema=True)
sales = spark.read.csv("../data/sales.csv", header=True, inferSchema=True)

print(f"Employees: {employees.count()} rows")
print(f"Departments: {departments.count()} rows")
print(f"Sales: {sales.count()} rows")

Employees: 30 rows
Departments: 6 rows
Sales: 100 rows


#### Try It: Read Sales with Explicit Schema

1. Define an explicit schema for `sales.csv` with: `sale_id` (IntegerType), `employee_id` (IntegerType), `product` (StringType), `amount` (FloatType), `quantity` (IntegerType), `sale_date` (StringType), `region` (StringType)
2. Read the file with this schema
3. Compare `printSchema()` output vs the inferred version above — what changed?

In [10]:
# Your code here


In [11]:
# --- Solution ---
from pyspark.sql.types import FloatType

sales_schema = StructType([
    StructField("sale_id", IntegerType(), False),
    StructField("employee_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("amount", FloatType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("sale_date", StringType(), True),
    StructField("region", StringType(), True),
])

sales_explicit = spark.read.csv("../data/sales.csv", header=True, schema=sales_schema)
print("Explicit schema:")
sales_explicit.printSchema()
print("\nInferred schema:")
sales.printSchema()
# Notice: amount is FloatType (explicit) vs DoubleType (inferred)

Explicit schema:
root
 |-- sale_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: float (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- region: string (nullable = true)


Inferred schema:
root
 |-- sale_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- sale_date: date (nullable = true)
 |-- region: string (nullable = true)



---
## Concept 4: Inspection Methods

Several methods help you understand what's inside a DataFrame without seeing all the data.

In [12]:
# Column names as a Python list
print(f"Columns: {employees.columns}")

# Column names + types as list of tuples
print(f"Types: {employees.dtypes}")

# Row count
print(f"Row count: {employees.count()}")

# First row as a Row object — access fields by name
first_row = employees.first()
print(f"First employee: {first_row['name']}, salary: {first_row['salary']}")

Columns: ['employee_id', 'name', 'department_id', 'salary', 'hire_date', 'city']
Types: [('employee_id', 'int'), ('name', 'string'), ('department_id', 'int'), ('salary', 'int'), ('hire_date', 'date'), ('city', 'string')]
Row count: 30
First employee: Alice Chen, salary: 92000


#### Try It: Inspect the Sales DataFrame

1. Print the column names of `sales`
2. Print the dtypes
3. Get the first row and print the `product` and `amount` fields

In [13]:
# Your code here


In [15]:
# --- Solution ---
print(f"Sales columns: {sales.columns}")
print(f"Sales dtypes: {sales.dtypes}")
first_sale = sales.first()
print(f"First sale: product={first_sale['product']}, amount={first_sale['amount']}")

Sales columns: ['sale_id', 'employee_id', 'product', 'amount', 'quantity', 'sale_date', 'region']
Sales dtypes: [('sale_id', 'int'), ('employee_id', 'int'), ('product', 'string'), ('amount', 'double'), ('quantity', 'int'), ('sale_date', 'date'), ('region', 'string')]
First sale: product=Widget A, amount=1250.0


---
## Concept 5: describe() and summary()

These methods compute **summary statistics** for numeric columns — count, mean, stddev, min, max. Essential for quick data exploration.

- `.describe()` — basic stats (count, mean, stddev, min, max)
- `.summary()` — extended stats (adds percentiles: 25%, 50%, 75%)

In [16]:
# Basic statistics
employees.describe().show()

+-------+-----------------+-------------+------------------+------------------+-------------+
|summary|      employee_id|         name|     department_id|            salary|         city|
+-------+-----------------+-------------+------------------+------------------+-------------+
|  count|               30|           30|                30|                30|           30|
|   mean|             15.5|         NULL|103.16666666666667| 81833.33333333333|         NULL|
| stddev|8.803408430829505|         NULL|1.7436255002314767|15168.213891018222|         NULL|
|    min|                1| Aaron Brooks|               101|             61000|      Chicago|
|    max|               30|Zara Mitchell|               106|            115000|San Francisco|
+-------+-----------------+-------------+------------------+------------------+-------------+



In [17]:
# Extended statistics with percentiles
employees.summary().show()

+-------+-----------------+-------------+------------------+------------------+-------------+
|summary|      employee_id|         name|     department_id|            salary|         city|
+-------+-----------------+-------------+------------------+------------------+-------------+
|  count|               30|           30|                30|                30|           30|
|   mean|             15.5|         NULL|103.16666666666667| 81833.33333333333|         NULL|
| stddev|8.803408430829505|         NULL|1.7436255002314767|15168.213891018222|         NULL|
|    min|                1| Aaron Brooks|               101|             61000|      Chicago|
|    25%|                8|         NULL|               102|             70000|         NULL|
|    50%|               15|         NULL|               103|             78000|         NULL|
|    75%|               23|         NULL|               105|             93000|         NULL|
|    max|               30|Zara Mitchell|               106|

#### Try It: Explore Sales Statistics

1. Run `.describe()` on the sales DataFrame
2. From the output, answer: What is the **max** sale amount? What is the **average** quantity?

In [18]:
# Your code here


In [19]:
# --- Solution ---
sales.describe().show()
# Look at the 'max' row for 'amount' and 'mean' row for 'quantity'

+-------+------------------+-----------------+--------+------------------+------------------+-------+
|summary|           sale_id|      employee_id| product|            amount|          quantity| region|
+-------+------------------+-----------------+--------+------------------+------------------+-------+
|  count|               100|              100|     100|               100|               100|    100|
|   mean|              50.5|            14.53|    NULL|           2068.27|              3.94|   NULL|
| stddev|29.011491975882016|8.779515874460222|    NULL|1929.4469783192937|3.1328174946117517|   NULL|
|    min|                 1|                1|Gadget X|             150.0|                 1|Central|
|    max|               100|               30|Widget C|            9600.0|                15|   West|
+-------+------------------+-----------------+--------+------------------+------------------+-------+



---
## Capstone Exercise

1. Define explicit schemas for ALL 3 CSV files (employees, departments, sales)
2. Read each file using its explicit schema
3. Print all 3 schemas side by side
4. Run `.describe()` on employees and sales
5. Print the first row of each DataFrame, accessing at least 2 fields by name

In [20]:
# Your capstone code here


In [21]:
# --- Capstone Solution ---
emp_schema = StructType([
    StructField("employee_id", IntegerType(), False),
    StructField("name", StringType(), True),
    StructField("department_id", IntegerType(), True),
    StructField("salary", IntegerType(), True),
    StructField("hire_date", StringType(), True),
    StructField("city", StringType(), True),
])

dept_schema = StructType([
    StructField("department_id", IntegerType(), False),
    StructField("department_name", StringType(), True),
    StructField("budget", IntegerType(), True),
    StructField("location", StringType(), True),
])

sales_schema = StructType([
    StructField("sale_id", IntegerType(), False),
    StructField("employee_id", IntegerType(), True),
    StructField("product", StringType(), True),
    StructField("amount", FloatType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("sale_date", StringType(), True),
    StructField("region", StringType(), True),
])

emp = spark.read.csv("../data/employees.csv", header=True, schema=emp_schema)
dept = spark.read.csv("../data/departments.csv", header=True, schema=dept_schema)
sal = spark.read.csv("../data/sales.csv", header=True, schema=sales_schema)

print("=== Employee Schema ===")
emp.printSchema()
print("=== Department Schema ===")
dept.printSchema()
print("=== Sales Schema ===")
sal.printSchema()

print("\n=== Employee Stats ===")
emp.describe().show()
print("=== Sales Stats ===")
sal.describe().show()

e1 = emp.first()
print(f"First employee: {e1['name']}, salary: {e1['salary']}")
d1 = dept.first()
print(f"First department: {d1['department_name']}, budget: {d1['budget']}")
s1 = sal.first()
print(f"First sale: {s1['product']}, amount: {s1['amount']}")

=== Employee Schema ===
root
 |-- employee_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department_id: integer (nullable = true)
 |-- salary: integer (nullable = true)
 |-- hire_date: string (nullable = true)
 |-- city: string (nullable = true)

=== Department Schema ===
root
 |-- department_id: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- budget: integer (nullable = true)
 |-- location: string (nullable = true)

=== Sales Schema ===
root
 |-- sale_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: float (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- sale_date: string (nullable = true)
 |-- region: string (nullable = true)


=== Employee Stats ===
+-------+-----------------+-------------+------------------+------------------+----------+-------------+
|summary|      employee_id|         name|     department_id|            salary| hire_dat

In [22]:
spark.stop()

---
## What You Learned

- Create DataFrames from **tuples** (with column names) or **dictionaries**
- **StructType/StructField** define explicit schemas — faster and more reliable than inference
- **`header=True`** and **`inferSchema=True`** are the key CSV reading options
- **`.columns`**, **`.dtypes`**, **`.count()`**, **`.first()`** for quick inspection
- **`.describe()`** and **`.summary()`** for statistical overview

Next: **Module 4 — DataFrame Transformations**, where you'll learn to actually transform and analyze data with select, filter, groupBy, join.